# Lab 07-01 — Recall@k, MRR, nDCG against the BEIR fiqa qrels

**Track 07 · Evaluation** — the three retrieval metrics every later lab reuses: did the retriever put the right documents near the top?

This notebook is **self-contained**: it imports LangChain and faiss directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS index, the top-10 retriever, and the three metric formulas all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

All three metrics answer "did the retriever put the right documents near the top?" — but they measure different failure modes:

* **Recall@k** — did the gold documents make it into the top-k *at all*? Ignores order inside the window. A recall ceiling means the retriever never even surfaces the answer.
* **MRR@10** — how quickly does the FIRST gold document appear? Rewards getting one right document to the very top, ignores the rest.
* **nDCG@10** — position-weighted overall ranking quality. Still rewards an early hit, but keeps counting the other gold documents further down.

This lab wires the three together on fiqa (BEIR financial FAQ corpus): the first 8,000 documents are indexed in FAISS with the local BGE embedder, then every test query whose gold documents are inside the subset is retrieved at top-10 and scored with all three metrics against the ground-truth qrels. The metric formulas are written out inline below — the formula is the point — instead of imported from the shared evaluation block.


## Setup

One prerequisite must hold before this notebook will run:

- **beir-fiqa on disk** — `Data/corpus/beir-fiqa/fiqa/` (BEIR financial FAQ corpus: `corpus.jsonl` + `queries.jsonl` + `qrels/test.tsv`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, and `faiss-cpu`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   sentence-transformers -> local BGE embeddings (HuggingFaceEmbeddings)
#   langchain-huggingface -> the HuggingFaceEmbeddings wrapper
#   langchain-community   -> the FAISS vector store
#   faiss-cpu             -> the FAISS index
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import math
import os
import sys
import time
from pathlib import Path

# LangChain + faiss — the only libraries this notebook needs. Nothing is
# imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `FIQA_DIR` points at the fiqa subset already on disk; `FIQA_N_DOCS = 8000` takes a deterministic head of the 57,638-doc corpus (no randomness, reproducible runs); `FIQA_MAX_QUERIES = 60` bounds the search pool to the first 60 qrels-covered queries whose gold documents are inside the subset; `EVAL_K = 10` is the ranking depth every metric is computed at; `BGE_MODEL_NAME` selects the local embedder.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
FIQA_DIR = Path("Data/corpus/beir-fiqa/fiqa")
CORPUS_PATH = FIQA_DIR / "corpus.jsonl"
QUERIES_PATH = FIQA_DIR / "queries.jsonl"
QRELS_PATH = FIQA_DIR / "qrels" / "test.tsv"
FIQA_N_DOCS = 8000  # deterministic head of the 57,638-doc fiqa corpus
FIQA_MAX_QUERIES = 60  # pool cap: first N qrels-covered queries with gold inside
EVAL_K = 10  # ranking depth every metric is computed at
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # local BGE embedder (cached)


## 2. Load — fiqa corpus + queries + qrels

BEIR datasets keep three files: `corpus.jsonl` (the documents), `queries.jsonl` (the questions), and `qrels/test.tsv` — one line per *judged* (query, document) pair with a relevance grade. Those qrels are the ground truth: for each query they say which documents *should* be retrieved. `load_corpus` reads the first `n` corpus docs as `title + " " + text` — fiqa titles are short keyword phrases the queries are written against, so they belong in the indexed text. `load_queries` returns `(query_id, query_text)` pairs in file order, and `load_qrels` keeps only the qrels rows with score >= 1 as `{query_id: {relevant_corpus_id, ...}}`. This is the same layout the nfcorpus track used, so the loader is reusable across BEIR sets.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — fiqa corpus + queries + qrels (BEIR layout, same as nfcorpus)
# --------------------------------------------------------------------------
def load_corpus(path: Path, n: int) -> tuple[list[str], list[str]]:
    """Return (doc_texts, doc_ids) for the first ``n`` corpus docs."""
    texts: list[str] = []
    ids: list[str] = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            doc = json.loads(line)
            ids.append(doc["_id"])
            texts.append(f'{doc["title"]} {doc["text"]}')
    return texts, ids


def load_queries(path: Path) -> list[tuple[str, str]]:
    """Return [(query_id, query_text)] for every query, in file order."""
    out: list[tuple[str, str]] = []
    with open(path) as f:
        for line in f:
            q = json.loads(line)
            out.append((q["_id"], q["text"]))
    return out


def load_qrels(path: Path) -> dict[str, set[str]]:
    """Return {query_id: {relevant_corpus_id, ...}} (qrels score >= 1)."""
    qrels: dict[str, set[str]] = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3 or parts[0] == "query-id":
                continue  # header row
            qid, cid = parts[0], parts[1]
            if int(parts[2]) >= 1:
                qrels.setdefault(qid, set()).add(cid)
    return qrels


## 3. Experiment — index, retrieve, score

One index, one retriever, three metrics — and this time the metric formulas are written out inline in the cell below, because the formula is the point of the lab:

* **Recall@k** — `|top-k ∩ gold| / |gold|`: of the relevant documents, what fraction did we get into the top-k? (Rank-insensitive.)
* **MRR@k** — `1 / rank` of the *first* relevant document within the top-k, else 0. Punishes a relevant document buried at position 8.
* **nDCG@k** — position-discounted gain: each gold id at 1-based position `i` contributes `1 / log2(i + 1)`, normalized by the ideal ranking (all gold at the top). Rewards the overall ordering.

Each answers a different production question: *do we cover the topic?* (Recall), *does the best hit surface early?* (MRR), *is the overall ordering good?* (nDCG). The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine); we embed the 8,000 docs once, then hand the FAISS store its vectors through a tiny precomputed passthrough — so the embed step and the index step stay separately timed, exactly like the lab. Retrieval is the store's own `as_retriever` at `EVAL_K = 10`, and each retrieved list is scored with all three metrics against the qrels.


In [ ]:
# --------------------------------------------------------------------------
# 3. Metrics + experiment — index the subset, retrieve top-10 per query, score
# --------------------------------------------------------------------------
def recall_at_k(ranked_ids: list[str], gold: set[str], k: int) -> float:
    """Fraction of the gold set found within the top ``k`` ranks.

    0.0 when ``gold`` is empty (nothing to recall) or ``k`` is 0. Otherwise
    ``|top-k ∩ gold| / |gold|``.
    """
    if not gold or k <= 0:
        return 0.0
    top_k = set(ranked_ids[:k])
    return len(top_k & gold) / len(gold)


def mrr_at_k(ranked_ids: list[str], gold: set[str], k: int) -> float:
    """Mean-reciprocal-rank-style score for one query: 1/rank of the first hit.

    Returns ``1 / rank`` (1-based) for the first gold id found within the top
    ``k`` positions, else 0.0. Called MRR only when averaged over queries;
    per-query it is the reciprocal rank.
    """
    for rank, cid in enumerate(ranked_ids[:k], start=1):
        if cid in gold:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ranked_ids: list[str], gold: set[str], k: int) -> float:
    """Discounted cumulative gain normalized by the ideal ranking.

    Binary relevance: each gold id in position ``i`` (1-based) contributes
    ``1 / log2(i + 1)``. The ideal is the same sum over the first ``min(k,
    |gold|)`` positions. Returns 0.0 when ``gold`` is empty; the ideal is
    truncated at ``k`` so scores land in [0, 1].
    """
    if not gold or k <= 0:
        return 0.0
    gain = 0.0
    for i, cid in enumerate(ranked_ids[:k], start=1):
        if cid in gold:
            gain += 1.0 / math.log2(i + 1)
    ideal = 0.0
    for i in range(1, min(k, len(gold)) + 1):
        ideal += 1.0 / math.log2(i + 1)
    return gain / ideal if ideal > 0.0 else 0.0


# Several track agents share this machine — cap BLAS/OpenMP threads so the
# BGE embedding step stays light on CPU and memory.
os.environ["OMP_NUM_THREADS"] = "2"
import torch  # noqa: E402
torch.set_num_threads(2)


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call. embed_query is delegated to the real embedder so
    the store's retriever can embed queries.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]],
                 query_embedder: Embeddings):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))
        self._query_embedder = query_embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._query_embedder.embed_query(text)


def run_experiment() -> dict:
    fiqa_texts, fiqa_ids = load_corpus(CORPUS_PATH, FIQA_N_DOCS)
    fiqa_queries = load_queries(QUERIES_PATH)
    qrels = load_qrels(QRELS_PATH)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    fiqa_vecs = embedder.embed_documents(fiqa_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": cid})
        for t, cid in zip(fiqa_texts, fiqa_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings(fiqa_texts, fiqa_vecs, embedder)
    )
    index_s = time.perf_counter() - t0

    retriever = store.as_retriever(search_kwargs={"k": EVAL_K})

    subset_ids = set(fiqa_ids)
    covered = [
        (qid, q) for qid, q in fiqa_queries
        if qid in qrels and qrels[qid] & subset_ids
    ][:FIQA_MAX_QUERIES]

    t0 = time.perf_counter()
    rows: list[dict] = []
    for qid, qtext in covered:
        gold = qrels[qid] & subset_ids
        ranked = [d.metadata["id"] for d in retriever.invoke(qtext)]
        rows.append({
            "qid": qid,
            "gold": sorted(gold),
            "recall_1": recall_at_k(ranked, gold, 1),
            "recall_5": recall_at_k(ranked, gold, 5),
            "recall_10": recall_at_k(ranked, gold, 10),
            "mrr_10": mrr_at_k(ranked, gold, 10),
            "ndcg_10": ndcg_at_k(ranked, gold, 10),
        })
    retrieve_s = time.perf_counter() - t0

    def mean(key: str) -> float:
        return sum(r[key] for r in rows) / len(rows) if rows else 0.0

    return {
        "rows": rows,
        "indexed": len(fiqa_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "retrieve_s": retrieve_s,
        "metrics": {
            "recall@1": mean("recall_1"),
            "recall@5": mean("recall_5"),
            "recall@10": mean("recall_10"),
            "mrr@10": mean("mrr_10"),
            "ndcg@10": mean("ndcg_10"),
        },
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with embedding/index timings; the mean retrieval metrics over the query pool; three example rows showing each query's gold set and its scores; then a takeaway explaining why the gap between recall@10 and recall@1 is the headroom that reranking (track 06) exists to harvest.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 07-01 — Recall@k, MRR, nDCG against the BEIR fiqa qrels")
    print(f"fiqa {exp['indexed']} docs, pool {len(exp['rows'])} queries, "
          f"k = {EVAL_K}")
    print("=" * 66)

    m = exp["metrics"]
    print(f"\n[1] Mean retrieval metrics over {len(exp['rows'])} queries:")
    for label in ("recall@1", "recall@5", "recall@10", "mrr@10", "ndcg@10"):
        print(f"    {label:10s}: {m[label]:.4f}")

    print(f"\n[2] Three example rows:")
    for row in exp["rows"][:3]:
        print(f"    {row['qid']}: gold={row['gold']}, R@5={row['recall_5']:.2f}, "
              f"MRR@10={row['mrr_10']:.2f}, nDCG@10={row['ndcg_10']:.2f}")

    print(f"\n[3] Timing: embed {exp['indexed']} docs {exp['embed_s']:.1f}s, "
          f"index {exp['index_s']:.1f}s, retrieve {exp['retrieve_s']:.1f}s")

    print(f"\n[4] Takeaway")
    print("    Recall and MRR/nDCG measure different things: recall@1 shows")
    print("    how often the BEST retriever position already holds a gold")
    print("    document, while nDCG@10 rewards the overall ordering of all")
    print("    gold documents. The gap between recall@10 and recall@1 is the")
    print("    headroom that reranking (track 06) exists to harvest.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `FIQA_N_DOCS` docs indexed, a query pool of at least 40, every metric in [0, 1], recall monotonicity (`R@10 >= R@5 >= R@1`), and that the retriever actually finds some gold (`recall@10 > 0`). This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    m = exp["metrics"]
    pool = len(exp["rows"])

    checks.append((f"exactly {FIQA_N_DOCS} fiqa docs indexed",
                   exp["indexed"] == FIQA_N_DOCS))
    checks.append((f"pool has {pool} queries (>= 40)", pool >= 40))

    for label in ("recall@1", "recall@5", "recall@10", "mrr@10", "ndcg@10"):
        checks.append((f"{label} in [0, 1]", 0.0 <= m[label] <= 1.0))

    checks.append(("recall monotonic: R@10 >= R@5 >= R@1",
                   m["recall@10"] >= m["recall@5"] >= m["recall@1"]))
    checks.append(("recall@10 > 0 (retriever actually finds gold)",
                   m["recall@10"] > 0.0))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Embedding 8,000 documents takes under a minute; the 60 retrievals after that are instant. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The scored table: mean Recall@1/5/10, MRR@10, and nDCG@10 over the query pool, plus three example rows. The interesting signal is the gap between Recall@1 and Recall@10 — retrieval usually covers the topic by k=10, but the *top slot* is where a RAG answer actually draws its context from.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the fiqa files are intact.


In [ ]:
verify_gate(exp)
